# Tools and tool calling

Assistant version 4: Answer order, shipment and stock questions through a hand-written tool loop.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --group authoring`, then `uv run jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## @tool; docstrings and type hints become JSON schema

Predict the output before running. The function definitions below match `examples/01_tool_schema.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Turn a typed function and its docstring into a tool contract.

Run: uv run python modules/05_tools/examples/01_tool_schema.py --offline
Expected: A JSON input schema and a stock record from the read-only database.
Cost: No model calls; database reads only.
"""

import json

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.tools import make_order_tools


def main() -> None:
    lesson_settings()
    stock = make_order_tools("C001")[2]
    # KEY IDEA: the model sees a schema, not the Python function's implementation.
    print_text(stock.description)
    print_text(json.dumps(stock.args_schema.model_json_schema(), indent=2))
    print_text(str(stock.invoke({"sku": "BW-001"})))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Pydantic args_schema; input validation; bounded tool errors

Predict the output before running. The function definitions below match `examples/02_args_schema.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Reject malformed tool arguments before any database operation.

Run: uv run python modules/05_tools/examples/02_args_schema.py --offline
Expected: Invalid SKU rejected, followed by a valid stock result.
Cost: No model calls.
"""

from pydantic import ValidationError

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.tools import make_order_tools


def main() -> None:
    lesson_settings()
    stock = make_order_tools("C001")[2]
    try:
        stock.invoke({"sku": "not-a-sku"})
    except ValidationError:
        print_text("Invalid SKU rejected before SQL execution.")
    # KEY IDEA: validation is a boundary, not a guarantee the SKU exists.
    print_text(str(stock.invoke({"sku": "BW-001"})))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## bind_tools; AIMessage.tool_calls; ToolMessage; matching tool_call_id

Predict the output before running. The function definitions below match `examples/03_dispatch.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Bind tools, inspect a model request and return its matching ToolMessage.

Run: uv run python modules/05_tools/examples/03_dispatch.py --offline
Expected: A stock request and a result with tool_call_id=call-1.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from common.agent import dispatch
from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message
from common.tools import make_order_tools


def main() -> None:
    settings = lesson_settings()
    tools = make_order_tools("C001")
    model = get_chat_model(
        settings=settings, responses=[tool_request("check_stock", {"sku": "BW-001"})]
    )
    reply = model.bind_tools(tools).invoke("Check stock for BW-001 using the stock tool.")
    print_message(reply)
    # KEY IDEA: binding does not execute Python. The allowlisted dispatcher does.
    for call in reply.tool_calls:
        print_message(dispatch(call, tools))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Parallel tool calls; tool_choice; provider support checks

Predict the output before running. The function definitions below match `examples/04_parallel_choice.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Dispatch independent read-only requests concurrently and examine tool_choice.

Run: uv run python modules/05_tools/examples/04_parallel_choice.py --offline
Expected: Two stock results with distinct call IDs in offline mode.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from concurrent.futures import ThreadPoolExecutor

from langchain_core.messages import AIMessage

from common.agent import dispatch
from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message, print_text
from common.tools import make_order_tools


def main() -> None:
    settings = lesson_settings()
    tools = make_order_tools("C001")
    fixture = AIMessage(
        content="",
        tool_calls=[
            {"name": "check_stock", "args": {"sku": sku}, "id": f"stock-{i}"}
            for i, sku in enumerate(["BW-001", "BW-002"])
        ],
    )
    model = get_chat_model(settings=settings, responses=[fixture])
    options = {} if settings.llm_provider == "ollama" else {"tool_choice": "check_stock"}
    if settings.llm_provider == "ollama":
        print_text("Ollama accepts but ignores tool_choice; the prompt requests stock explicitly.")
    reply = model.bind_tools(tools, **options).invoke(
        "Check stock for BW-001 and BW-002. Request both if supported."
    )
    # KEY IDEA: only independent reads are safe to execute concurrently here.
    with ThreadPoolExecutor(max_workers=2) as pool:
        for result in pool.map(lambda call: dispatch(call, tools), reply.tool_calls):
            print_message(result)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Build it yourself: bounded tool-calling loop; stop condition and unknown-tool rejection

Predict the output before running. The function definitions below match `examples/05_manual_agent.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Write the model-tool-model loop by hand with strict limits.

Run: uv run python modules/05_tools/examples/05_manual_agent.py --offline
Expected: A stock tool result followed by a final fixture reply.
Cost: At most four model turns and eight tool calls; offline/local no API fee.
"""

from langchain_core.messages import HumanMessage

from common.agent import dispatch
from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message
from common.tools import make_order_tools


def main() -> None:
    settings = lesson_settings()
    tools = make_order_tools("C001")
    model = get_chat_model(
        settings=settings,
        responses=[
            tool_request("check_stock", {"sku": "BW-001"}),
            "The stock result is shown above; stock is not reserved.",
        ],
    )
    bound = model.bind_tools(tools)
    messages = [HumanMessage("Use the tool to check BW-001 stock.")]
    calls = 0
    for _ in range(4):
        reply = bound.invoke(messages)
        if reply.invalid_tool_calls:
            raise ValueError("Invalid tool response.")
        messages.append(reply)
        print_message(reply)
        if not reply.tool_calls:
            return
        calls += len(reply.tool_calls)
        if calls > 8:
            raise RuntimeError("Tool-call budget exhausted.")
        # KEY IDEA: each ToolMessage completes one pending model-requested call.
        for call in reply.tool_calls:
            result = dispatch(call, tools)
            messages.append(result)
            print_message(result)
    raise RuntimeError("Turn budget exhausted.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## ToolRuntime; runtime state and context; preview with a minimal harness

Predict the output before running. The function definitions below match `examples/06_runtime.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Inspect ToolRuntime context and state using an explicit injection harness.

Run: uv run python modules/05_tools/examples/06_runtime.py --offline
Expected: Customer C001 and two supplied messages; runtime is absent from the public tool schema.
Cost: No model call. Module 10 replaces the harness with agent-managed injection.
"""

from dataclasses import dataclass

from langchain.tools import ToolRuntime, tool
from langchain_core.messages import HumanMessage

from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text


@dataclass(frozen=True)
class CustomerContext:
    customer_id: str


@tool
def session_summary(runtime: ToolRuntime[CustomerContext]) -> str:
    """Read the trusted session identity and current message count."""
    return f"Customer {runtime.context.customer_id}; messages={len(runtime.state['messages'])}"


def main() -> None:
    lesson_settings()
    runtime = ToolRuntime(
        state={"messages": [HumanMessage("Hello"), HumanMessage("Order status?")]},
        context=CustomerContext("C001"),
        config={},
        stream_writer=lambda _: None,
        tool_call_id="runtime-1",
        store=None,
    )
    # KEY IDEA: normally the agent injects runtime. This harness supplies it explicitly.
    print_text(session_summary.invoke({"runtime": runtime}))
    print_text(str(session_summary.tool_call_schema.model_json_schema()))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Parameterized SQL; read-only database connection; customer scope; filesystem/code tool risks

Predict the output before running. The function definitions below match `examples/07_sql_safety.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Enforce ownership and reject SQL-shaped arguments through constrained tools.

Run: uv run python modules/05_tools/examples/07_sql_safety.py --offline
Expected: Owned order found, another customer denied, malformed ID rejected.
Cost: No model calls; all database connections are read-only.
"""

from common.agent import dispatch
from common.errors import run_cli
from common.lesson import lesson_settings
from common.pretty import print_text
from common.tools import make_order_tools


def main() -> None:
    lesson_settings()
    tools = make_order_tools("C001")
    for order_id in ("BW1001", "BW1002", "BW1001' OR 1=1 --"):
        result = dispatch(
            {"name": "lookup_order", "args": {"order_id": order_id}, "id": "safe-1"}, tools
        )
        print_text(result.content)
    # KEY IDEA: SQL placeholders handle values; a separate customer predicate enforces scope.
    print_text("No arbitrary SQL, shell, filesystem or Python-execution tool is exposed.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Answer order, shipment and stock questions through a hand-written tool loop

Predict the output before running. The function definitions below match `use_case/order_status.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Answer order-status questions using the audited, bounded manual tool loop.

Run: uv run python modules/05_tools/use_case/order_status.py --offline
Expected: Order and shipment results for C001/BW1001, then a draft response.
Cost: At most four model calls, usually under $0.01; offline/local no API fee.
"""

from common.agent import run_tool_loop
from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message
from common.tools import make_order_tools


def main() -> None:
    settings = lesson_settings()
    # Synthetic demo login is trusted application context; never read identity from a prompt.
    tools = make_order_tools("C001")
    model = get_chat_model(
        settings=settings,
        responses=[
            tool_request("lookup_order", {"order_id": "BW1001"}, "order-1"),
            tool_request("get_shipment_status", {"order_id": "BW1001"}, "shipment-1"),
            "Order BW1001 is delivered. "
            "Please review the shipment record above for its delivery date.",
        ],
    )
    for message in run_tool_loop(
        model, tools, "Please look up order BW1001 and its shipment status."
    )[2:]:
        print_message(message)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run pytest modules/05_tools/tests` for the supplied solution checks.